# NumPy Numerical Analysis: South Africa Food Security

Group Members:
1. Sibusiso Agent Mathonsi
2. Kegoikantse Sebetseba
3. Agcobile Qabo
4. Lebogang Malatjie
5. Tlotlo Naledi
6. Tlotlanang Naledi
7. Thandi Sebokolodi

This notebook supports the numerical analysis section of the project. It uses the cleaned datasets from `Data Preparation`, converts important numeric columns into NumPy arrays, calculates descriptive and comparative results, reshapes arrays, and writes summary outputs for the report.

## 1. Load Cleaned Data

The analysis starts from the cleaned CSV files so that all numerical work is based on validated South Africa records.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Numeric Analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PREP_DIR = PROJECT_ROOT / "Data Preparation"
OUTPUT_DIR = PROJECT_ROOT / "Numeric Analysis"
OUTPUT_DIR.mkdir(exist_ok=True)

svfi = pd.read_csv(DATA_PREP_DIR / "cleaned_severe_food_insecurity_south_africa.csv").sort_values("year")
ipc = pd.read_csv(DATA_PREP_DIR / "cleaned_ipc_south_africa.csv")
phase_dist = pd.read_csv(DATA_PREP_DIR / "cleaned_ipc_phase_distribution_south_africa.csv")

print("Severe food insecurity cleaned data:", svfi.shape)
print("IPC cleaned data:", ipc.shape)
print("IPC phase distribution cleaned data:", phase_dist.shape)

Severe food insecurity cleaned data: (6, 15)
IPC cleaned data: (12, 20)
IPC phase distribution cleaned data: (10, 21)


## 2. Convert Columns To NumPy Arrays

The key numeric columns are converted from pandas Series into NumPy arrays. This makes it clear where NumPy is being used for the numerical calculations.

In [2]:
years = svfi["year"].to_numpy(dtype=int)
prevalence = svfi["obs_value"].to_numpy(dtype=float)

people = phase_dist[phase_dist["metric"].eq("people")].sort_values("phase_order").copy()
percentages = phase_dist[phase_dist["metric"].eq("percentage")].sort_values("phase_order").copy()

phase_labels = people["phase"].to_numpy()
phase_orders = people["phase_order"].to_numpy(dtype=float)
people_values = people["obs_value"].to_numpy(dtype=float)
percentage_values = percentages["obs_value"].to_numpy(dtype=float)

print("Years array:", years)
print("Severe food insecurity prevalence array:", prevalence)
print("IPC people array:", people_values)
print("IPC percentage array:", percentage_values)

Years array: [2018 2019 2020 2021 2022 2023]
Severe food insecurity prevalence array: [6.9 7.5 8.  8.4 8.4 8.5]
IPC people array: [34950493. 14844785.  8175072.  1160087.       nan]
IPC percentage array: [59. 25. 14.  2. nan]


## 3. Severe Food Insecurity Trend Calculations

These calculations summarise the prevalence trend using NumPy totals, averages, minimums, maximums, ranges, and changes over time. The prevalence values are percentages, so the total is a sum of annual observations rather than a population total.

In [3]:
svfi_total_observation_sum = np.sum(prevalence)
svfi_average = np.mean(prevalence)
svfi_minimum = np.min(prevalence)
svfi_maximum = np.max(prevalence)
svfi_range = np.ptp(prevalence)
svfi_total_change = prevalence[-1] - prevalence[0]
svfi_relative_change = (svfi_total_change / prevalence[0]) * 100

yoy_changes = np.diff(prevalence)
relative_changes = np.divide(
    yoy_changes,
    prevalence[:-1],
    out=np.full_like(yoy_changes, np.nan, dtype=float),
    where=prevalence[:-1] != 0,
) * 100

svfi_stats = pd.DataFrame(
    [
        {"metric": "Observation count", "value": prevalence.size, "unit": "records"},
        {"metric": "Sum of annual prevalence observations", "value": round(svfi_total_observation_sum, 2), "unit": "percent observation sum"},
        {"metric": "Average prevalence", "value": round(svfi_average, 2), "unit": "percent"},
        {"metric": "Minimum prevalence", "value": round(svfi_minimum, 2), "unit": "percent"},
        {"metric": "Maximum prevalence", "value": round(svfi_maximum, 2), "unit": "percent"},
        {"metric": "Range", "value": round(svfi_range, 2), "unit": "percentage points"},
        {"metric": "Total change", "value": round(svfi_total_change, 2), "unit": "percentage points"},
        {"metric": "Relative change from first year", "value": round(svfi_relative_change, 2), "unit": "percent change"},
        {"metric": "Average year-on-year change", "value": round(np.mean(yoy_changes), 2), "unit": "percentage points"},
    ]
)
svfi_stats

,metric,value,unit
0,Observation count,6.00,records
1,Sum of annual prevalence observations,47.70,percent observation sum
2,Average prevalence,7.95,percent
3,Minimum prevalence,6.90,percent
4,Maximum prevalence,8.50,percent
5,Range,1.60,percentage points
6,Total change,1.60,percentage points
7,Relative change from first year,23.19,percent change
8,Average year-on-year change,0.32,percentage points


**Meaning for the project story:** severe food insecurity increased across the available years. The latest value is higher than the first value, and the second half of the period is higher on average than the first half.

## 4. Compare Values Across Time Periods

`np.diff()` calculates year-on-year percentage-point changes. A relative percentage change is also calculated to show how large each movement is compared with the previous year's value.

In [4]:
yoy_table = pd.DataFrame(
    {
        "from_year": years[:-1],
        "to_year": years[1:],
        "from_value_percent": np.round(prevalence[:-1], 2),
        "to_value_percent": np.round(prevalence[1:], 2),
        "change_percentage_points": np.round(yoy_changes, 2),
        "relative_change_percent": np.round(relative_changes, 2),
    }
)

yoy_table.to_csv(OUTPUT_DIR / "severe_food_insecurity_year_on_year_change.csv", index=False)
yoy_table

,from_year,to_year,from_value_percent,to_value_percent,change_percentage_points,relative_change_percent
0,2018,2019,6.9,7.5,0.6,8.70
1,2019,2020,7.5,8.0,0.5,6.67
2,2020,2021,8.0,8.4,0.4,5.00
3,2021,2022,8.4,8.4,0.0,0.00
4,2022,2023,8.4,8.5,0.1,1.19


## 5. Reshape Arrays

The six-year prevalence array can be reshaped into two 3-year periods. This demonstrates NumPy reshaping and supports a simple comparison between the earlier and later period.

In [5]:
trend_pair_matrix = np.column_stack((years, prevalence))
trend_period_matrix = prevalence.reshape(2, 3)
period_averages = np.mean(trend_period_matrix, axis=1)
period_difference = period_averages[1] - period_averages[0]

period_comparison = pd.DataFrame(
    {
        "period": [f"{years[0]}-{years[2]}", f"{years[3]}-{years[-1]}"],
        "values_percent": [list(np.round(trend_period_matrix[0], 2)), list(np.round(trend_period_matrix[1], 2))],
        "average_percent": np.round(period_averages, 2),
    }
)

print("Year/prevalence matrix shape:", trend_pair_matrix.shape)
print("Reshaped 3-year period matrix shape:", trend_period_matrix.shape)
print("Later period minus earlier period:", round(period_difference, 2), "percentage points")
period_comparison

Year/prevalence matrix shape: (6, 2)
Reshaped 3-year period matrix shape: (2, 3)
Later period minus earlier period: 0.97 percentage points


,period,values_percent,average_percent
0,2018-2020,"[6.9, 7.5, 8.0]",7.47
1,2021-2023,"[8.4, 8.4, 8.5]",8.43


**Meaning for the project story:** after reshaping the trend into two equal 3-year periods, the later period has the higher average prevalence. This supports the story that severe food insecurity was more intense in the later available years.

## 6. IPC Phase Distribution Calculations

The IPC phase data gives a snapshot of how many people were classified in each food insecurity phase in 2020-10. NumPy is used to sum phase rows and compare lower phases with crisis-or-worse phases.

In [6]:
phase_metric_matrix = np.vstack((people_values, percentage_values))

p3plus_people = float(
    ipc.loc[(ipc["indicator"].eq("IPC_IPC_P3PLUS")) & (ipc["metric"].eq("people")), "obs_value"]
    .dropna()
    .iloc[0]
)
p3plus_percent = float(
    ipc.loc[(ipc["indicator"].eq("IPC_IPC_P3PLUS")) & (ipc["metric"].eq("percentage")), "obs_value"]
    .dropna()
    .iloc[0]
)

reported_phase_people = np.nansum(people_values)
reported_phase_percentage = np.nansum(percentage_values)
crisis_mask = phase_orders >= 3
lower_phase_mask = phase_orders <= 2
phase_row_crisis_people = np.nansum(people_values[crisis_mask])
phase_row_crisis_percent = np.nansum(percentage_values[crisis_mask])
lower_phase_people = np.nansum(people_values[lower_phase_mask])
lower_phase_percent = np.nansum(percentage_values[lower_phase_mask])
missing_phase_values = np.isnan(phase_metric_matrix).sum()
people_range = np.nanmax(people_values) - np.nanmin(people_values)

ipc_stats = pd.DataFrame(
    [
        {"metric": "Reported phase-row population", "value": round(reported_phase_people, 0), "unit": "people"},
        {"metric": "Reported phase-row share", "value": round(reported_phase_percentage, 2), "unit": "percent"},
        {"metric": "Source-reported Phase 3+ people", "value": round(p3plus_people, 0), "unit": "people"},
        {"metric": "Source-reported Phase 3+ percentage", "value": round(p3plus_percent, 2), "unit": "percent"},
        {"metric": "Phase 3-4 people from phase rows", "value": round(phase_row_crisis_people, 0), "unit": "people"},
        {"metric": "Phase 1-2 people", "value": round(lower_phase_people, 0), "unit": "people"},
        {"metric": "Phase 1-2 percentage", "value": round(lower_phase_percent, 2), "unit": "percent"},
        {"metric": "Range across reported phase people counts", "value": round(people_range, 0), "unit": "people"},
        {"metric": "Missing phase observations", "value": int(missing_phase_values), "unit": "records"},
    ]
)

print("IPC phase metric matrix shape:", phase_metric_matrix.shape)
ipc_stats

IPC phase metric matrix shape: (2, 5)


,metric,value,unit
0,Reported phase-row population,59130437.0,people
1,Reported phase-row share,100.0,percent
2,Source-reported Phase 3+ people,9335159.0,people
3,Source-reported Phase 3+ percentage,16.0,percent
4,Phase 3-4 people from phase rows,9335159.0,people
5,Phase 1-2 people,49795278.0,people
6,Phase 1-2 percentage,84.0,percent
7,Range across reported phase people counts,33790406.0,people
8,Missing phase observations,2.0,records


**Meaning for the project story:** the IPC snapshot shows that most classified people were in Phase 1 or Phase 2, but 9.3 million people were still in Phase 3 or above. Phase 5 remains missing in the source and should not be interpreted as zero famine.

## 7. Create Report-Ready Summary Tables

These tables are saved into the `Numeric Analysis` folder for use in the report, database section, and presentation evidence.

In [7]:
summary_rows = [
    {"metric": "Severe food insecurity observations analyzed", "value": int(prevalence.size), "unit": "records"},
    {"metric": "Severe food insecurity prevalence sum of annual observations", "value": round(float(np.sum(prevalence)), 2), "unit": "percent observation sum"},
    {"metric": "Severe food insecurity prevalence average", "value": round(float(np.mean(prevalence)), 2), "unit": "percent"},
    {"metric": "Severe food insecurity prevalence minimum", "value": round(float(np.min(prevalence)), 2), "unit": "percent"},
    {"metric": "Severe food insecurity prevalence maximum", "value": round(float(np.max(prevalence)), 2), "unit": "percent"},
    {"metric": "Severe food insecurity prevalence range", "value": round(float(np.ptp(prevalence)), 2), "unit": "percentage points"},
    {"metric": "Severe food insecurity total change", "value": round(float(prevalence[-1] - prevalence[0]), 2), "unit": "percentage points"},
    {"metric": "Severe food insecurity relative change from first year", "value": round(float((prevalence[-1] - prevalence[0]) / prevalence[0] * 100), 2), "unit": "percent change"},
    {"metric": "Severe food insecurity average year-on-year change", "value": round(float(np.mean(yoy_changes)), 2), "unit": "percentage points"},
    {"metric": "Severe food insecurity largest year-on-year increase", "value": round(float(np.max(yoy_changes)), 2), "unit": "percentage points"},
    {"metric": "Severe food insecurity first reshaped 3-year average", "value": round(float(period_averages[0]), 2), "unit": "percent"},
    {"metric": "Severe food insecurity second reshaped 3-year average", "value": round(float(period_averages[1]), 2), "unit": "percent"},
    {"metric": "Severe food insecurity reshaped-period difference", "value": round(float(period_difference), 2), "unit": "percentage points"},
    {"metric": "IPC reported phase-row population, 2020-10", "value": round(float(reported_phase_people), 0), "unit": "people"},
    {"metric": "IPC reported phase-row share, 2020-10", "value": round(float(reported_phase_percentage), 2), "unit": "percent"},
    {"metric": "IPC source-reported people in Phase 3 or above, 2020-10", "value": round(float(p3plus_people), 0), "unit": "people"},
    {"metric": "IPC source-reported percentage in Phase 3 or above, 2020-10", "value": round(float(p3plus_percent), 2), "unit": "percent"},
    {"metric": "Reported Phase 3-4 people from phase rows, 2020-10", "value": round(float(phase_row_crisis_people), 0), "unit": "people"},
    {"metric": "Reported Phase 3-4 share from phase rows, 2020-10", "value": round(float(phase_row_crisis_percent), 2), "unit": "percent"},
    {"metric": "IPC people in Phase 1-2, 2020-10", "value": round(float(lower_phase_people), 0), "unit": "people"},
    {"metric": "IPC percentage in Phase 1-2, 2020-10", "value": round(float(lower_phase_percent), 2), "unit": "percent"},
    {"metric": "IPC people count range across reported phases", "value": round(float(people_range), 0), "unit": "people"},
    {"metric": "IPC missing phase observations retained", "value": int(missing_phase_values), "unit": "records"},
]

numerical_summary = pd.DataFrame(summary_rows)
numerical_summary.to_csv(OUTPUT_DIR / "numerical_summary.csv", index=False)

phase_summary = people[["phase", "phase_order", "obs_value", "obs_status_label"]].merge(
    percentages[["phase", "phase_order", "obs_value", "obs_status_label"]],
    on=["phase", "phase_order"],
    suffixes=("_people", "_percentage"),
)
phase_summary.rename(
    columns={
        "obs_value_people": "people",
        "obs_status_label_people": "people_status",
        "obs_value_percentage": "percentage",
        "obs_status_label_percentage": "percentage_status",
    },
    inplace=True,
)
phase_summary["people"] = phase_summary["people"].round(0)
phase_summary["percentage"] = phase_summary["percentage"].round(1)
phase_summary["reporting_treatment"] = np.where(
    phase_summary[["people", "percentage"]].isna().any(axis=1),
    "Not reported in source",
    "Reported",
)
phase_summary["people_display"] = phase_summary["people"].map(
    lambda value: "Not reported" if pd.isna(value) else f"{value:,.0f}"
)
phase_summary["percentage_display"] = phase_summary["percentage"].map(
    lambda value: "Not reported" if pd.isna(value) else f"{value:.1f}%"
)
phase_summary = phase_summary.sort_values("phase_order").reset_index(drop=True)
phase_summary.to_csv(OUTPUT_DIR / "ipc_phase_summary.csv", index=False)

print("Wrote numerical_summary.csv")
print("Wrote severe_food_insecurity_year_on_year_change.csv")
print("Wrote ipc_phase_summary.csv")
numerical_summary

Wrote numerical_summary.csv
Wrote severe_food_insecurity_year_on_year_change.csv
Wrote ipc_phase_summary.csv


,metric,value,unit
0,Severe food insecurity observations analyzed,6.00,records
1,Severe food insecurity prevalence sum of annua...,47.70,percent observation sum
2,Severe food insecurity prevalence average,7.95,percent
3,Severe food insecurity prevalence minimum,6.90,percent
4,Severe food insecurity prevalence maximum,8.50,percent
5,Severe food insecurity prevalence range,1.60,percentage points
6,Severe food insecurity total change,1.60,percentage points
7,Severe food insecurity relative change from fi...,23.19,percent change
8,Severe food insecurity average year-on-year ch...,0.32,percentage points
9,Severe food insecurity largest year-on-year in...,0.60,percentage points


## 8. Write Findings Text

The findings file explains what the numerical outputs mean for the project story.

In [8]:
largest_yoy_index = int(np.argmax(yoy_changes))
smallest_yoy_index = int(np.argmin(yoy_changes))

findings = [
    "NUMERICAL FINDINGS",
    "",
    "- Loaded the cleaned severe food insecurity and IPC datasets from the Data Preparation folder, then converted years, prevalence values, IPC people counts, and IPC percentages into NumPy arrays.",
    f"- The severe food insecurity prevalence array covers {years[0]} to {years[-1]} and contains {prevalence.size} yearly observations.",
    f"- Prevalence increased from {prevalence[0]:.1f}% in {years[0]} to {prevalence[-1]:.1f}% in {years[-1]}, a total change of {prevalence[-1] - prevalence[0]:.1f} percentage points, or {((prevalence[-1] - prevalence[0]) / prevalence[0] * 100):.2f}% relative to the first year.",
    f"- Across the severe food insecurity series, the NumPy average is {np.mean(prevalence):.2f}%, the minimum is {np.min(prevalence):.1f}%, the maximum is {np.max(prevalence):.1f}%, and the range is {np.ptp(prevalence):.1f} percentage points.",
    f"- The largest year-on-year increase is {yoy_changes[largest_yoy_index]:.1f} percentage points from {years[largest_yoy_index]} to {years[largest_yoy_index + 1]}; the smallest increase is {yoy_changes[smallest_yoy_index]:.1f} percentage points from {years[smallest_yoy_index]} to {years[smallest_yoy_index + 1]}.",
    f"- Reshaping the six-year prevalence array into two 3-year periods gives a {years[0]}-{years[2]} average of {period_averages[0]:.2f}% and a {years[3]}-{years[-1]} average of {period_averages[1]:.2f}%, so the later period is {period_difference:.2f} percentage points higher.",
    f"- In the IPC 2020-10 snapshot, NumPy sums show {reported_phase_people:,.0f} people across reported phase rows and {reported_phase_percentage:.1f}% across reported phase percentages.",
    f"- The source-reported Phase 3 or above KPI is {p3plus_people:,.0f} people, equal to {p3plus_percent:.1f}% of the classified population; the Phase 3 and Phase 4 rows sum to the same people count because Phase 5 is missing in the supplied phase rows.",
    f"- Phase 1 and Phase 2 account for {lower_phase_people:,.0f} people, or {lower_phase_percent:.1f}%, meaning most classified people were outside crisis-or-worse phases in the IPC snapshot, while 16.0% were still in serious acute food insecurity.",
    f"- The IPC phase metric matrix was reshaped to {phase_metric_matrix.shape[0]} rows by {phase_metric_matrix.shape[1]} columns: one row for people and one row for percentages across the five IPC phases.",
    "- Phase 5 values are retained as missing values, not treated as zero, because the cleaned source labels them as missing.",
]

(OUTPUT_DIR / "numerical_findings.txt").write_text("\n".join(findings), encoding="utf-8")
print("Wrote numerical_findings.txt")

Wrote numerical_findings.txt


## Outputs Created

- `numpy_analysis.ipynb`
- `numerical_summary.csv`
- `severe_food_insecurity_year_on_year_change.csv`
- `ipc_phase_summary.csv`
- `numerical_findings.txt`